# Explicit bin edges

A size distribution is only as readable as its bins. An archived product that stores bin midpoints
alone leaves the edges to be guessed, and a guessed edge moves every concentration per unit size.
The engine keeps each probe's edges as an explicit array of `num_bins + 1` values in millimetres:
`bin_edges`, `bin_width` and `resolution_um` in `arrayprobe_uiops.constants`. This notebook reads
them for five probe types, checks that widths and midpoints follow from the edges, and re-bins a
seeded random count array onto a common grid without losing a particle. It runs in seconds and
needs no data files.

**Before this:** [06_read_products](../1_getting_started/06_read_products.ipynb) (Reading the output files). **Related:** [02_probe_overlap_check](../7_analysis_and_outputs/02_probe_overlap_check.ipynb). **Needs:** core package only.

This notebook reads files from the arrayprobe source repository, so run it from a clone of that repository; `pip install arrayprobe` alone does not include them.

In [1]:
# Launch guard: the cells below reach scripts/ and tests/data/ relative to the repository root.
from pathlib import Path

if not (Path("scripts").is_dir() and Path("tests/data").is_dir()):
    raise RuntimeError(
        f"This notebook must run with the repository root as its working directory, not {Path.cwd()}. "
        "Jupyter starts a kernel in the notebook's own folder (examples/6_science/), so run "
        "`%cd ../..` in a new cell, then run this cell again. "
        "Headless, from the repository root: python scripts/execute_example_notebooks.py --check <name>"
    )

## 1. Edges for each probe type

Each table gives one more edge than there are bins, strictly increasing, and `bin_width` equals
the difference of consecutive edges.

In [2]:
import numpy as np
from arrayprobe_uiops.constants import bin_edges, bin_width, resolution_um

PROJECT = "demo"  # any project name without its own table uses the general one
PROBES = ["2DS", "HVPS", "CIP", "2DC", "PIP"]

for probe in PROBES:
    e = bin_edges(PROJECT, probe)
    w = bin_width(PROJECT, probe)
    assert e.ndim == 1 and len(w) == len(e) - 1, probe
    assert np.all(np.diff(e) > 0), probe
    assert np.allclose(np.diff(e), w), probe
    print(f"{probe:5s} {len(e) - 1:4d} bins  {e[0]:.3f}-{e[-1]:.3f} mm  "
          f"pixel {resolution_um(PROJECT, probe):.0f} um  first width {w[0] * 1000:.0f} um")

2DS    128 bins  0.005-1.285 mm  pixel 10 um  first width 10 um
HVPS    28 bins  0.200-30.000 mm  pixel 150 um  first width 200 um
CIP     19 bins  0.050-2.000 mm  pixel 25 um  first width 50 um
2DC     19 bins  0.050-2.000 mm  pixel 30 um  first width 50 um
PIP     64 bins  0.050-6.450 mm  pixel 100 um  first width 100 um


## 2. Midpoints come from the edges, never the reverse

The midpoint of each bin lies strictly inside it. Going the other way is not possible in general:
on a grid whose widths change, the midpoints do not determine the edges, which is why the edges are
the stored quantity.

In [3]:
for probe in PROBES:
    e = bin_edges(PROJECT, probe)
    mid = 0.5 * (e[:-1] + e[1:])
    assert np.all((mid > e[:-1]) & (mid < e[1:])), probe

# Two different edge sets with the same midpoints: the midpoints alone are ambiguous.
a = np.array([0.0, 1.0, 3.0, 7.0])
b = np.array([-0.5, 1.5, 2.5, 7.5])
mid_a, mid_b = 0.5 * (a[:-1] + a[1:]), 0.5 * (b[:-1] + b[1:])
assert np.allclose(mid_a, mid_b) and not np.allclose(a, b)
print("midpoints", mid_a, "fit both edge sets", a, "and", b)

midpoints [0.5 2.  5. ] fit both edge sets [0. 1. 3. 7.] and [-0.5  1.5  2.5  7.5]


## 3. Concentration per unit size round-trips through the widths

Dividing counts by the bin width gives a per-millimetre distribution; multiplying back by the same
widths recovers the counts exactly. A wrong width would scale each bin by a different factor.

In [4]:
rng = np.random.default_rng(44)
e2ds = bin_edges(PROJECT, "2DS")
w2ds = bin_width(PROJECT, "2DS")
counts = rng.poisson(lam=200 * np.exp(-np.arange(len(w2ds)) / 15.0)).astype(float)

per_mm = counts / w2ds
assert np.allclose(per_mm * w2ds, counts)
print(f"{counts.sum():.0f} particles in {len(w2ds)} bins")

3087 particles in 128 bins


## 4. Re-binning onto a common grid keeps every particle

To compare probes, counts are moved onto a shared grid by the overlap of each source bin with each
target bin, assuming particles spread evenly within a source bin. With explicit edges this is exact
bookkeeping: a target grid that spans the source range receives every particle.

In [5]:
def rebin(counts, src_edges, dst_edges):
    lo = np.maximum(src_edges[:-1, None], dst_edges[None, :-1])
    hi = np.minimum(src_edges[1:, None], dst_edges[None, 1:])
    overlap = np.clip(hi - lo, 0.0, None) / np.diff(src_edges)[:, None]
    return counts @ overlap

common = np.geomspace(e2ds[0], e2ds[-1], 21)  # 20 log-spaced bins over the same range
moved = rebin(counts, e2ds, common)
assert np.isclose(moved.sum(), counts.sum())

# Within each common bin, the count equals the source counts that fall inside it.
assert np.all(moved >= 0)
print(f"before {counts.sum():.0f}, after {moved.sum():.6f} particles")

before 3087, after 3087.000000 particles


A target grid narrower than the source loses exactly the particles outside it, and the edges say
how many:

In [6]:
narrow = np.array([0.1, 0.2, 0.4, 0.8])
kept = rebin(counts, e2ds, narrow).sum()
inside = rebin(counts, e2ds, np.array([0.1, 0.8])).sum()
assert np.isclose(kept, inside) and kept < counts.sum()
print(f"{kept:.1f} of {counts.sum():.0f} particles lie between 0.1 and 0.8 mm")

1611.5 of 3087 particles lie between 0.1 and 0.8 mm


## Summary

- Every probe table carries `num_bins + 1` strictly increasing edges in millimetres, and its widths
  are their differences.
- Midpoints follow from the edges; the edges cannot be recovered from midpoints in general.
- Per-millimetre concentrations round-trip through the widths exactly.
- Re-binning by overlap onto a grid that spans the source range conserves the particle count, and a
  narrower grid loses exactly the particles outside it.